# 🇮🇳 India EV Adoption Analysis
## Notebook 01 — Data Cleaning & Preparation

**Project:** Predicting ICE-to-EV Transition Across Indian States  
**Author:** [Your Name]  
**Dataset:** India EV Market Data (2018–2024)

---

### 🎯 What this notebook does:
1. Loads all 4 raw CSV files
2. Audits each dataset (shape, nulls, types)
3. Cleans each dataset individually
4. Merges them into one master dataframe
5. Saves cleaned files for use in the next notebooks

### 💡 Why we clean before anything else?
Garbage in = garbage out. If your data has missing values, wrong mappings, or inconsistent formats,
your analysis will be wrong — and your ML model will learn from wrong patterns.
Cleaning first ensures every downstream step is built on solid ground.

---
## SECTION 1: Imports & Setup

In [13]:
# Standard data science imports
# pandas  → working with tabular data (like Excel but in Python)
# numpy   → numerical operations
# matplotlib/seaborn → plotting
# os      → file path handling

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')  # suppress minor warnings for cleaner output


pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

print('✅ Libraries loaded successfully')

✅ Libraries loaded successfully


In [15]:
# ── File Paths ────────────────────────────────────────────────────────
# WHY: Defining paths as variables means if you move files,
#      you only change one line instead of hunting through all your code.

DATA_RAW   = 'data/raw/'       # original files — never modify these
DATA_CLEAN = 'data/processed/' # cleaned outputs go here

# Create output folder if it doesn't exist
os.makedirs(DATA_CLEAN, exist_ok=True)

print(f'📁 Raw data folder:     {DATA_RAW}')
print(f'📁 Processed folder:    {DATA_CLEAN}')

📁 Raw data folder:     data/raw/
📁 Processed folder:    data/processed/


---
## SECTION 2: Load Raw Data

In [21]:
# Run this before changing directory
import os
print(os.listdir('data'))

['processed']


In [23]:
# Load all 4 datasets
# WHY keep them separate initially: each needs its own cleaning logic.
# We'll merge them at the end of this notebook.

sales    = pd.read_csv('data/raw/EV_Ice_Market_Sales_India.csv')
charging = pd.read_csv('data/raw/EV_Charging_Infrastructure_India.csv')
policy   = pd.read_csv('data/raw/EV_Policy_Incentives_India.csv')
battery  = pd.read_csv('data/raw/Vehicle_Battery_Performance_India.csv')

print('Datasets loaded:')
print(f'  sales    → {sales.shape[0]} rows × {sales.shape[1]} cols')
print(f'  charging → {charging.shape[0]} rows × {charging.shape[1]} cols')
print(f'  policy   → {policy.shape[0]} rows × {policy.shape[1]} cols')
print(f'  battery  → {battery.shape[0]} rows × {battery.shape[1]} cols')

Datasets loaded:
  sales    → 147 rows × 6 cols
  charging → 600 rows × 7 cols
  policy   → 28 rows × 5 cols
  battery  → 120 rows × 7 cols


---
## SECTION 3: Audit Each Dataset

**WHY audit before cleaning?**  
You can't fix what you haven't diagnosed. Auditing tells you:
- How many missing values exist and in which columns
- Whether data types are correct (e.g. is 'year' stored as int or string?)
- Whether values make logical sense (e.g. EV sales > total sales would be a bug)

Think of this like a doctor doing tests before prescribing medicine.

In [27]:
# ── Helper function: audit any dataframe ─────────────────────────────
# WHY a function: we need to do the same audit on 4 datasets.
# Repeating code 4 times is bad practice → wrap it in a function.

def audit(df, name):
    print(f'{'='*55}')
    print(f'📋 AUDIT: {name}')
    print(f'{'='*55}')
    print(f'Shape         : {df.shape[0]} rows × {df.shape[1]} columns')
    print(f'Duplicate rows: {df.duplicated().sum()}')
    print()
    print('Column info:')
    info = pd.DataFrame({
        'dtype':    df.dtypes,
        'nulls':    df.isnull().sum(),
        'null_%':   (df.isnull().sum() / len(df) * 100).round(1),
        'unique':   df.nunique()
    })
    print(info)
    print()
    print('First 3 rows:')
    display(df.head(3))
    print()

In [29]:
audit(sales, 'EV_Ice_Market_Sales_India')

📋 AUDIT: EV_Ice_Market_Sales_India
Shape         : 147 rows × 6 columns
Duplicate rows: 0

Column info:
                  dtype  nulls  null_%  unique
year              int64      0    0.00       7
state            object      0    0.00       7
vehicle_segment  object      0    0.00       3
ev_sales          int64      0    0.00     144
ice_sales         int64      0    0.00     147
total_sales       int64      0    0.00     147

First 3 rows:


,year,state,vehicle_segment,ev_sales,ice_sales,total_sales
0,2018,Maharashtra,2W,19738,108340,128078
1,2018,Maharashtra,3W,9266,66276,75542
2,2018,Maharashtra,4W,6180,45190,51370


In [ ]:
audit(charging, 'EV_Charging_Infrastructure_India')

In [ ]:
audit(policy, 'EV_Policy_Incentives_India')

In [ ]:
audit(battery, 'Vehicle_Battery_Performance_India')

---
## SECTION 4: Clean Dataset 1 — Sales Data

**Issues found in audit:**
- No missing values ✅
- Need to add derived columns (ev_share_pct, yoy_growth)
- Need to verify: ev_sales + ice_sales == total_sales (data integrity check)

**Key concept — Feature Engineering:**  
Raw columns rarely tell the full story. `ev_sales = 19,738` means nothing unless you
know it's out of 128,000 total sales. Creating `ev_share_pct = 15.4%` is what makes
the data comparable across states of different sizes.

In [ ]:
# ── Step 1: Data Integrity Check ──────────────────────────────────────
# Does ev_sales + ice_sales always equal total_sales?
# If not, there's a calculation error in the source data.

sales_check = sales['ev_sales'] + sales['ice_sales']
mismatch = (sales_check != sales['total_sales']).sum()
print(f'Rows where ev + ice ≠ total: {mismatch}')

# WHY: Trust but verify. Always sanity-check relationships between columns.

In [ ]:
# ── Step 2: Feature Engineering ───────────────────────────────────────
# These new columns are the core metrics of our entire analysis.

# EV Share % — what % of total vehicle sales were EVs?
# WHY: This normalises across states (Maharashtra has way more sales than Telangana)
sales['ev_share_pct'] = (sales['ev_sales'] / sales['total_sales'] * 100).round(2)

# ICE Share % — complement of EV share
sales['ice_share_pct'] = (sales['ice_sales'] / sales['total_sales'] * 100).round(2)

# Year-over-Year EV growth — how much did EV share change vs previous year?
# WHY: Raw share tells you current state; YoY growth tells you momentum/velocity
# sort_values ensures we calculate YoY in chronological order per group
sales = sales.sort_values(['state', 'vehicle_segment', 'year'])
sales['ev_share_yoy'] = (
    sales.groupby(['state', 'vehicle_segment'])['ev_share_pct']
    .pct_change() * 100
).round(2)
# pct_change() computes (current - previous) / previous * 100
# The first year (2018) per group will be NaN — that's expected and fine

# Absolute EV growth — raw unit increase year-over-year
sales['ev_sales_yoy'] = sales.groupby(['state', 'vehicle_segment'])['ev_sales'].diff()

print('New columns added:', ['ev_share_pct', 'ice_share_pct', 'ev_share_yoy', 'ev_sales_yoy'])
print()
display(sales.head(10))

In [ ]:
# ── Step 3: Quick sanity check on our new columns ─────────────────────
print('EV share % range:', sales['ev_share_pct'].min(), 'to', sales['ev_share_pct'].max())
print('Sum of EV + ICE share (should always be 100):')
print((sales['ev_share_pct'] + sales['ice_share_pct']).describe())

# If max and min are both 100.0, we're good.

In [ ]:
# ── Step 4: State-level aggregation ───────────────────────────────────
# Create a state-year level summary (collapsing across vehicle segments)
# WHY: Many of our visualisations and ML features will work at state-year level,
#      not at the segment level. We need both granularities.

sales_state = sales.groupby(['year', 'state']).agg(
    total_ev_sales   = ('ev_sales',     'sum'),
    total_ice_sales  = ('ice_sales',    'sum'),
    total_sales      = ('total_sales',  'sum')
).reset_index()

sales_state['ev_share_pct'] = (sales_state['total_ev_sales'] / sales_state['total_sales'] * 100).round(2)

print('State-year aggregated sales:')
display(sales_state.head(10))

---
## SECTION 5: Clean Dataset 2 — Charging Infrastructure

**Issues found in audit:**
- 276 nulls in `avg_daily_sessions` (46% missing!) — needs imputation
- **CRITICAL BUG:** City-state mismatches (Ahmedabad listed under Maharashtra, Delhi listed under UP, etc.) — the city and state columns appear randomly assigned, not accurately mapped

**Decision on city-state mismatch:**  
Since city data is unreliable but state data seems intentional (the 7 states match our sales data), we'll **trust the state column and drop the city column** for analysis purposes. We use state as our join key.

In [ ]:
# ── Step 1: Handle avg_daily_sessions nulls ───────────────────────────
# WHY median and not mean?
# Mean is sensitive to outliers. If one supercharger has 500 sessions/day,
# it pulls the mean up for everyone. Median is more robust.
# WHY group by charger_type?
# Fast DC chargers likely have different usage patterns than Slow AC.
# Filling Fast DC nulls with a Slow AC median would be misleading.

charging['avg_daily_sessions'] = charging.groupby('charger_type')['avg_daily_sessions'].transform(
    lambda x: x.fillna(x.median())
)

print('Nulls after imputation:', charging['avg_daily_sessions'].isnull().sum())
print()
print('Median sessions per charger type:')
print(charging.groupby('charger_type')['avg_daily_sessions'].median())

In [ ]:
# ── Step 2: Drop city column (unreliable), keep state ─────────────────
charging = charging.drop(columns=['city'])
print('City column dropped. Remaining columns:', list(charging.columns))

In [ ]:
# ── Step 3: Create state-level infrastructure summary ─────────────────
# WHY: Our ML model works at state level. We need to summarise
#      600 individual station rows into one row per state.

charging_state = charging.groupby('state').agg(
    total_stations       = ('station_id',          'count'),
    fast_dc_stations     = ('charger_type',         lambda x: (x == 'Fast DC').sum()),
    slow_ac_stations     = ('charger_type',         lambda x: (x == 'Slow AC').sum()),
    avg_capacity_kw      = ('charger_capacity_kw',  'mean'),
    avg_daily_sessions   = ('avg_daily_sessions',   'mean'),
    urban_stations       = ('area_type',            lambda x: (x == 'Urban').sum()),
    rural_stations       = ('area_type',            lambda x: (x == 'Rural').sum())
).reset_index()

# Derived ratios — more meaningful than raw counts
charging_state['fast_dc_ratio']    = (charging_state['fast_dc_stations'] / charging_state['total_stations'] * 100).round(1)
charging_state['urban_ratio']      = (charging_state['urban_stations']   / charging_state['total_stations'] * 100).round(1)

print('Charging infrastructure per state:')
display(charging_state)

---
## SECTION 6: Clean Dataset 3 — Policy Incentives

**Issues found:**
- 19 nulls in `incentive_amount_rs` (68% missing) — too many to impute reliably
- Data is at policy level (4 rows per state)

**Strategy:**  
Since the rupee amounts are mostly missing, we'll focus on what IS reliable:
- **Policy count per state** — how many policies does a state have?
- **Earliest policy year** — did early adopters of policy see faster EV growth?
- **Policy type mix** — what types of incentives are active?

This is a real-world data science decision: when a column is too sparse to use directly, engineer new features from what IS available.

In [ ]:
# ── Step 1: Engineer policy features per state ────────────────────────

policy_state = policy.groupby('state').agg(
    policy_count         = ('policy_name',          'count'),
    earliest_policy_year = ('launch_year',           'min'),
    latest_policy_year   = ('launch_year',           'max'),
    avg_incentive_rs     = ('incentive_amount_rs',   'mean'),    # uses only non-null rows
    has_fame_subsidy     = ('policy_name',           lambda x: int('FAME II Subsidy' in x.values)),
    has_road_tax_exempt  = ('policy_name',           lambda x: int('Road Tax Exemption' in x.values)),
    central_policies     = ('policy_type',           lambda x: (x == 'Central').sum()),
    state_policies       = ('policy_type',           lambda x: (x == 'State').sum())
).reset_index()

# Policy age — how many years has the state had at least one policy? (as of 2024)
# WHY: A state that adopted policy in 2019 has had 5 years of incentive effect vs
#      one that adopted in 2023 (only 1 year). This captures policy maturity.
policy_state['policy_age_years'] = 2024 - policy_state['earliest_policy_year']

# Fill avg_incentive_rs with 0 for states with all-null incentive amounts
policy_state['avg_incentive_rs'] = policy_state['avg_incentive_rs'].fillna(0).round(0)

print('Policy features per state:')
display(policy_state)

---
## SECTION 7: Clean Dataset 4 — Battery Performance

**Issues found:**  
- No missing values ✅  
- No state column — this dataset is about vehicle models, not regions  
- Used for: understanding the product landscape, not for state-level ML model

**What we'll do:**  
Create segment-level summaries (avg price, range, battery size per 2W/3W/4W).
These become features we can attach to the sales data by vehicle segment.

In [ ]:
# ── Segment-level battery/product summary ─────────────────────────────
battery_segment = battery.groupby('vehicle_segment').agg(
    model_count          = ('vehicle_model',            'count'),
    avg_battery_kwh      = ('battery_capacity_kwh',     'mean'),
    avg_range_km         = ('claimed_range_km',         'mean'),
    avg_charging_hrs     = ('charging_time_hours',      'mean'),
    avg_price_lakh       = ('ex_showroom_price_lakh',   'mean'),
    min_price_lakh       = ('ex_showroom_price_lakh',   'min'),
    max_price_lakh       = ('ex_showroom_price_lakh',   'max')
).reset_index().round(2)

print('Battery/product summary per segment:')
display(battery_segment)

print()
print('Manufacturer breakdown:')
display(battery.groupby(['vehicle_segment', 'manufacturer']).size().unstack(fill_value=0))

---
## SECTION 8: Build the Master Dataset

**The big merge — joining all cleaned datasets together.**

Think of it like Excel VLOOKUP but for multiple tables at once.

**Join strategy:**
- Start with `sales_state` (state × year level) as the backbone
- LEFT JOIN `charging_state` on `state` — adds infrastructure features
- LEFT JOIN `policy_state` on `state` — adds policy features

**WHY LEFT JOIN and not INNER JOIN?**  
Inner join would drop any state not present in ALL datasets.  
Left join keeps all rows from the left (sales) table — we never lose data.

In [ ]:
# ── Build master dataframe ─────────────────────────────────────────────

master = sales_state.copy()

# Join charging infrastructure (state level)
master = master.merge(charging_state, on='state', how='left')
print(f'After joining charging data: {master.shape}')

# Join policy data (state level)
master = master.merge(policy_state, on='state', how='left')
print(f'After joining policy data:   {master.shape}')

print()
print('Master dataset columns:')
print(list(master.columns))

In [ ]:
# ── Add time-aware policy features ────────────────────────────────────
# A key insight: policy effect isn't static — it grows over time.
# For each year in our data, we compute how long the policy had been active.
# e.g. In 2021, a policy launched in 2019 has been active for 2 years.

master['years_since_first_policy'] = (master['year'] - master['earliest_policy_year']).clip(lower=0)
# clip(lower=0) ensures we don't get negative values for years before policy launch

# Has the state launched at least one policy by this year?
master['policy_active'] = (master['year'] >= master['earliest_policy_year']).astype(int)

print('Time-aware policy features added.')
display(master[['year', 'state', 'ev_share_pct', 'total_stations',
                'policy_count', 'years_since_first_policy', 'policy_active']].head(14))

In [ ]:
# ── Final null check on master ─────────────────────────────────────────
print('Null counts in master dataset:')
nulls = master.isnull().sum()
print(nulls[nulls > 0])
print()
print(f'Total shape: {master.shape}')

In [ ]:
# ── Sort master for readability ────────────────────────────────────────
master = master.sort_values(['state', 'year']).reset_index(drop=True)

print('Master dataset preview:')
display(master.head(14))

---
## SECTION 9: Quick Visual Sanity Check

Before saving, always **plot your key metric** to confirm the data makes sense.  
If EV share shows a sudden spike or drop that looks wrong, investigate before moving on.

In [ ]:
# EV share over time per state — our most important metric
fig, ax = plt.subplots(figsize=(12, 5))

for state in master['state'].unique():
    data = master[master['state'] == state]
    ax.plot(data['year'], data['ev_share_pct'], marker='o', label=state, linewidth=2)

ax.set_title('EV Market Share (%) by State — 2018 to 2024', fontsize=14, fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('EV Share (%)')
ax.legend(loc='upper left', bbox_to_anchor=(1, 1))
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('outputs/charts/01_ev_share_over_time.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved.')

---
## SECTION 10: Save Cleaned Files

In [ ]:
# Save all cleaned datasets
# WHY save to CSV and not keep in memory?
# Each notebook should be independently runnable. Notebook 02 will simply
# load these files without needing to re-run all of notebook 01.

os.makedirs('outputs/charts', exist_ok=True)

master.to_csv(DATA_CLEAN + 'master_ev_data.csv', index=False)
sales.to_csv(DATA_CLEAN + 'sales_clean.csv', index=False)
sales_state.to_csv(DATA_CLEAN + 'sales_state_clean.csv', index=False)
charging_state.to_csv(DATA_CLEAN + 'charging_state_clean.csv', index=False)
policy_state.to_csv(DATA_CLEAN + 'policy_state_clean.csv', index=False)
battery.to_csv(DATA_CLEAN + 'battery_clean.csv', index=False)
battery_segment.to_csv(DATA_CLEAN + 'battery_segment_clean.csv', index=False)

print('✅ All files saved to data/processed/')
print()
print('Files created:')
for f in os.listdir(DATA_CLEAN):
    df_temp = pd.read_csv(DATA_CLEAN + f)
    print(f'  {f:45s} → {df_temp.shape[0]} rows × {df_temp.shape[1]} cols')

---
## ✅ Notebook 01 Summary

**What we did:**

| Step | Action | Why |
|------|--------|-----|
| Audit | Checked shape, nulls, dtypes for all 4 datasets | Never clean blindly |
| Sales | Added ev_share_pct, ice_share_pct, yoy_growth | Raw counts aren't comparable across states |
| Charging | Imputed nulls with group median | More accurate than global median |
| Charging | Dropped unreliable city column | Bad data is worse than no data |
| Charging | Aggregated 600 rows → 7 state summaries | ML needs one row per state |
| Policy | Engineered policy features from sparse data | 68% null amount column can't be used directly |
| Master | Merged all 4 datasets on state/year | One clean table for analysis + ML |

**Next:** `02_exploratory_analysis.ipynb` — visualise patterns, find correlations, tell the data story.

---
*💡 Commit this notebook to GitHub after running it successfully.*